# PCS (patent→paper) Hit Probability — citation percentile within FoS × year cohorts

Computes each paper's **citation percentile** within its **Field-of-Study × publication-year**
cohort, for each window (C3 / C5 / C10 / C_all), using **patent->paper (PCS) citations**. A percentile of 0.99 = top 1%,
0.95 = top 5%, 0.90 = top 10%. Primary key: `paper_id`.

## Raw / input data (directory & structure)
```
/project/jevans/Dawoon/Science of Science/pcs/output/pcs_citation.parquet   # paper_id, C_3, C_5, C_10, C_all (patent->paper (PCS) citations)
/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_metadata.parquet   # paper_id, year, FoS_0, FoS_rep
```
Cohort **FoS** = the paper's primary level-0 field: `FoS_rep` (max-score field) when present, else the
first field of `FoS_0`. Join to the citation counts is done out-of-core with **DuckDB** (metadata is 249.8M rows).

## Metric (definition)
Within each (FoS, year) cohort, `pctl_{w}` is the citation **percentile** in `[0, 1]` (higher = more
cited; zero-citation ties near 0, most-cited near 1). Read hits as: top 1% ⇔ `pctl ≥ 0.99`,
top 5% ⇔ `≥ 0.95`, top 10% ⇔ `≥ 0.90`. Computed for `C_3, C_5, C_10, C_all`.

## Output
`/project/jevans/Dawoon/Science of Science/pcs/output/pcs_hit_probability.parquet` — `paper_id, FoS, year` +
`pctl_{w}` for w ∈ {c3,c5,c10,call} (citation percentile in [0,1] within the cohort).

In [1]:
import os, sys
import numpy as np, pandas as pd, duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/pcs')
import pcs_common as pcs

# Was: an os.chdir('..') hunt for Data/SciSciNet_v2, then relative notebook/*/output paths.
#   notebook/pcs/output/pcs_citation.parquet     -> Science of Science/pcs/output
#   notebook/paper/output/paper_metadata.parquet -> Science of Science/OpenAlex/output
ROOT   = pcs.BASE
CIT    = pcs.out('pcs_citation.parquet')
META   = pcs.PAPER_META
OUT_FP = pcs.out('pcs_hit_probability.parquet')
pcs.preflight('pcs_hit_probability')

pcs base   : /project/jevans/Dawoon/Science of Science/pcs
output     : /project/jevans/Dawoon/Science of Science/pcs/output

  pcs_hit_probability   OK


In [2]:
WINS = [('c3', 'C_3'), ('c5', 'C_5'), ('c10', 'C_10'), ('call', 'C_all')]
def add_percentile(df, cohort_cols):
    """Within each cohort partition, the citation PERCENTILE per window in [0, 1]
    (higher = more cited). rank(ascending=True, method='min') puts zero-citation ties near 0 and the
    top near 1; e.g. top 1% / 5% / 10% correspond to pctl >= 0.99 / 0.95 / 0.90."""
    for wtag, wcol in WINS:
        df[f'pctl_{wtag}'] = df.groupby(cohort_cols)[wcol].rank(pct=True, ascending=True, method='min').astype('float32')
    return df

## 1. Join citations with FoS + year (DuckDB, out-of-core)

In [3]:
%%time
con = duckdb.connect(); con.execute("SET memory_limit='20GB'")
# primary FoS = FoS_rep, else first element of FoS_0
df = con.execute(f'''
SELECT c.paper_id,
       coalesce(m.FoS_rep, split_part(m.FoS_0, ';', 1)) AS FoS,
       m.year,
       c.C_3, c.C_5, c.C_10, c.C_all
FROM read_parquet('{CIT}') c
JOIN read_parquet('{META}') m USING (paper_id)
''').fetchdf()
df = df.dropna(subset=['FoS', 'year'])
df['year'] = df['year'].astype(int)
for c in ['C_3', 'C_5', 'C_10', 'C_all']:
    df[c] = df[c].fillna(0).astype('int64')
print(f'papers with FoS + year: {len(df):,}')

papers with FoS + year: 5,022,564


## 2. Compute percentiles + save

In [4]:
%%time
df = add_percentile(df, ['FoS', 'year'])
out = df[['paper_id', 'FoS', 'year'] + [f'pctl_{w}' for w in ['c3','c5','c10','call']]]
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
print('citation percentile summary:')
print(out[[f'pctl_{w}' for w in ['c3','c5','c10','call']]].describe().round(4).to_string())
display(out.head(8))

WROTE /project/jevans/Dawoon/Science of Science/pcs/output/pcs_hit_probability.parquet  (5,022,564 rows, 7 cols)
citation percentile summary:
            pctl_c3       pctl_c5      pctl_c10     pctl_call
count  5.022564e+06  5.022564e+06  5.022564e+06  5.022564e+06
mean   7.740000e-02  1.894000e-01  3.361000e-01  3.838000e-01
std    2.466000e-01  3.417000e-01  3.532000e-01  3.550000e-01
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  2.000000e-04  2.530000e-02
50%    1.000000e-04  2.000000e-04  2.438000e-01  3.866000e-01
75%    3.000000e-04  4.530000e-02  6.670000e-01  7.165000e-01
max    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00


,paper_id,FoS,year,pctl_c3,pctl_c5,pctl_c10,pctl_call
0,W2400511492,"Biochemistry, Genetics and Molecular Biology",1996,0.000043,0.000043,0.646226,0.313136
1,W2294787856,Engineering,1975,0.000274,0.000274,0.000274,0.522870
2,W2798442775,Computer Science,2018,0.000044,0.356599,0.032170,0.032170
3,W574138238,Engineering,1994,0.000073,0.000073,0.422382,0.010100
4,W572475561,Engineering,1973,0.000340,0.000340,0.000340,0.000340
5,W582570849,Earth and Planetary Sciences,1990,0.001905,0.001905,0.001905,0.009524
6,W389329709,Engineering,2011,0.998966,0.997459,0.996129,0.993529
7,W388699498,Engineering,2006,0.000029,0.000029,0.801706,0.741508
